# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026

### ⚙️ Before Running:
1. **Add Data** (right panel) — attach 4 datasets ✅ already done
2. Set **Accelerator → GPU T4 x2**
3. Set **Persistence → Files**
4. Run cells **top to bottom**

## Step 0: Inspect All Input Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=3):
    if depth > max_depth or not os.path.exists(path):
        return
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

explore('/kaggle/input', max_depth=2)

## Step 1: Clone GitHub Repo

In [ ]:
import os
if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    !cd /kaggle/working/DermaFusion-AI && git pull
os.chdir('/kaggle/working/DermaFusion-AI')
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
!pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — go to Settings > Accelerator > GPU T4 x2')

## Step 4: Link All Datasets → data/ folder
Kaggle stores datasets under `competitions/{slug}/` and `datasets/{username}/{slug}/`.
This cell automatically finds all of them.

In [ ]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

# Build search list: competitions/, then datasets/{username}/ for each user, then flat input/
search_roots = ['/kaggle/input/competitions']

# CRITICAL FIX: Kaggle datasets/ has an extra username level: datasets/{username}/{slug}/
datasets_dir = '/kaggle/input/datasets'
if os.path.exists(datasets_dir):
    for username in os.listdir(datasets_dir):
        user_path = os.path.join(datasets_dir, username)
        if os.path.isdir(user_path):
            search_roots.append(user_path)  # e.g. /kaggle/input/datasets/kmader/

search_roots.append('/kaggle/input')  # flat fallback

print('🔍 Searching in:', search_roots)

def find_folder(keywords):
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for folder in os.listdir(root):
            full = os.path.join(root, folder)
            if os.path.isdir(full) and any(kw in folder.lower() for kw in keywords):
                return full
    return None

def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.exists(dst):
        print(f'✅ {label}: already linked')
        return
    if src:
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
    else:
        print(f'❌ {label}: NOT FOUND')

# HAM10000 — /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/
link(find_folder(['ham10000', 'skin-cancer-mnist']), 'ham10000', 'HAM10000')

# ISIC 2019 — /kaggle/input/datasets/andrewmvd/isic-2019/
link(find_folder(['isic-2019', 'isic2019', 'skin-lesion-images']), 'isic_2019', 'ISIC 2019')

# ISIC 2020 — /kaggle/input/competitions/siim-isic-melanoma-classification/
link(find_folder(['siim-isic', 'melanoma-classification', 'isic-2020']), 'isic_2020', 'ISIC 2020')

# ISIC 2024 — /kaggle/input/competitions/isic-2024-challenge/
link(find_folder(['isic-2024', 'skin-cancer-detection-3d']), 'isic_2024', 'ISIC 2024')

print('\n📁 data/ contents:', os.listdir(data_dir))

## Step 5: Phase 1 — Train Segmentation (25 epochs)
Trains **Swin-UNet** to produce lesion masks.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Segmentation training complete!')

## Step 6: Phase 2 — Train Classifier (25 epochs)
Trains **EVA-02 Large + ConvNeXt V2** dual-branch fusion.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Classifier training complete!')

## Step 7: Save Weights to Output

In [ ]:
import shutil, os
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            shutil.copy(os.path.join(weights_src, f), '/kaggle/working/')
            print(f'✅ Saved: {f}  ({os.path.getsize("/kaggle/working/"+f)/1e6:.0f} MB)')
    print('\n📦 Download from the Output tab on the right.')
else:
    print('❌ No weights — did training complete?')

## Step 8: Full Evaluation (Metrics + GradCAM++)

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete!')